__Dataset__: [Link](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [1]:
import numpy as np
import pandas as pd
import re
import string
from utils.constants import chat_words
import os

import nltk
NLTK_DIR = os.path.abspath("./env/nltk_data")
# Tokenizer
nltk.download('punkt', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
nltk.download('punkt_tab', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# Correct spelling
nltk.download('words', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# stopword removal
nltk.download('stopwords', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# download wordnet
nltk.download('wordnet', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)

nltk.data.path.insert(0, NLTK_DIR)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams
from nltk.corpus import words, stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

from collections import Counter

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

In [3]:
DATAPATH = 'data/spam.csv'

dataset_v1 = pd.read_csv(
    DATAPATH,
    header=0,
    usecols=[0, 1],
    names=["label", "message"],
    encoding="utf-8",
    encoding_errors="ignore"
)

df = dataset_v1.copy()
df = df.dropna(subset=["label", "message"]).copy()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["message"] = df["message"].astype(str)

print(f"Messages: {len(df):,}")
print("Label counts:", df["label"].value_counts().to_dict())
df.sample(5)


Messages: 5,572
Label counts: {'ham': 4825, 'spam': 747}


,label,message
5097,ham,Sorry about that this is my mates phone and i ...
4360,ham,"Don't Think About \What u Have Got\"" Think Abo..."
1003,ham,&lt;#&gt; am I think? Should say on syllabus
569,ham,WOT U WANNA DO THEN MISSY?
2834,ham,"Ya they are well and fine., BBD(pooja) full pi..."


In [4]:
def remove_number_middle_of_text(text):
    pattern = re.compile(r'[a-zA-Z]+')
    text = pattern.sub(r"", text)
    return " ".join(text)

def remove_htmltags(text):
    pattern = re.compile(r"<.*?>")
    return pattern.sub(r"", text)

def remove_urls(text):
    pattern = re.compile(r"https?:\/\/\S+|www\.\S+")
    return pattern.sub(r"", text)

def remove_punctuations(text):
    excluded_punctuations = string.punctuation
    return text.translate(str.maketrans(" ", " ", excluded_punctuations))

def eloborate_chat_conversion(texts):
    new_text = []
    token_words = texts.split()
    
    for word in token_words:
        key = word.upper()
        if key in chat_words:
            new_text.append(chat_words[key])
        else:
            new_text.append(word)
    return " ".join(new_text)


def corrected_spelling(texts):
    results = []
    correct_words = [w.lower() for w in words.words()]
    correct_set = set(correct_words)
    text_tokens = texts.split()
    
    for word in text_tokens:
        lower_word = word.lower()
        # keep the correctly spelled words
        if lower_word in correct_set or not lower_word.isalpha() or len(lower_word) < 2:
            results.append(lower_word)
            continue
        
        incorrect_bigrams = set(ngrams(lower_word, 2))
        best_score, best_match = 1.0, word
        
        for w in correct_words:
            if w[0] == lower_word[0]:
                correct_bigrams = set(ngrams(w, 2))
                score = jaccard_distance(incorrect_bigrams, correct_bigrams)
                if score < best_score:
                    best_score, best_match = score, w
        results.append(best_match)
    return " ".join(results)

def remove_stopwords(texts):
    cleaned = []
    text_tokens = texts.split()
    stop_words = set(stopwords.words('english'))
    for word in text_tokens:
        # word is already lowered
        if word and word not in stop_words:
            cleaned.append(word)
    return " ".join(cleaned)


def remove_emoji(texts):
    text_tokens = texts.split()
    text = " ".join(text_tokens)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    texts = emoji_pattern.sub(r'', text)
    return texts


def stemming_words(text_tokens):
    ps = PorterStemmer()
    stemmed_words = []
    return [stemmed_words.append(ps.stem(word)) for word in text_tokens]

def lemmatize_words(text_tokens):
    lemmatized_words = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for word in text_tokens:
        lemmatized_words.append(wordnet_lemmatizer.lemmatize(word, pos='v'))
    return lemmatized_words


def text_preprocessing(text):
    # lowercasing
    text = text.lower()
        
    # remove html tags
    text = remove_htmltags(text)
    
    # remove urls from text
    text = remove_urls(text)
    
    text = remove_punctuations(text)
    
    text = eloborate_chat_conversion(text)
    
    text = corrected_spelling(text)
    
    text = remove_stopwords(text)
    
    text = remove_emoji(text)
    
    tokenized_text = word_tokenize(text)
    
    # tokenized_text = stemming_words(tokenized_text)
    
    tokenized_text = lemmatize_words(tokenized_text)
    
    
    return tokenized_text

In [5]:
# text_preprocessing("""helo amzing ADIH <p>hello world</p> <br/> https://helo_texts.com
# https://www.helo_texts.com
# http://localhost:5000"""+df['message'][0])
df['tokens'] = df['message'].apply(text_preprocessing)
# corpus_tokens = df['tokens'].tolist()

#### For deleting the memory using gc

In [6]:
# import gc

# # del X_train
# gc.collect()

In [7]:
df['cleaned_text'] = df['tokens'].apply(' '.join)
# df['cleaned_text'].iloc[4138]

### Features and targets

In [10]:
X = df['cleaned_text']
y = df['label']

In [11]:
X[0]

'go juror point crazy available bugi n great world la e buffet cine get amores wat'

#### Train and Test

In [12]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (4457,)
y_train: (4457,)
X_test: (1115,)
y_test: (1115,)


#### Convert target to numerical

In [13]:
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)

## Train

In [82]:
# np.vstack(document_to_onehot(X_train[:2]))

x = []
for tokens in X_train[:2]:
    x.append(document_to_onehot(tokens))

len(x[0])

45

#### OneHotEncoder

In [14]:
all_train_tokens = [[tok] for tokens in X_train.tolist() for tok in tokens if len(tok) > 0]   # shape (n_tokens, 1)

ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore', dtype=np.int8)
ohe.fit(all_train_tokens)

vocab_size = len(ohe.categories_[0])

def document_to_onehot(tokens, ohe=ohe):
    if not tokens:
        return np.zeros(len(ohe.categories_[0]), dtype=np.int8)
    rows = ohe.transform([[t] for t in tokens])
    return (rows.sum(axis=0).A1 > 0).astype(np.int8)

# ohe_mat = document_to_onehot(corpus_tokens, ohe)

# df['ohe'] = df['tokens'].apply(document_to_onehot)
X_train_ohe = []
X_test_ohe = []
for tokens in X_train.tolist():
    X_train_ohe.append(document_to_onehot(tokens))

for tokens in X_test.tolist():
    X_test_ohe.append(document_to_onehot(tokens))
print(f"After apply the OneHotEncoder: {len(X_train_ohe), len(X_test_ohe)}")


After apply the OneHotEncoder: (4457, 1115)


#### Bag Of Words

In [15]:
train_corpus = X_train
test_corpus = X_test

cv = CountVectorizer()
cv_ngram = CountVectorizer(ngram_range=(1, 2))
X_train_bow = cv.fit_transform(train_corpus)
X_test_bow = cv.transform(test_corpus)

X_train_bow_ngram = cv_ngram.fit_transform(train_corpus)
X_test_bow_ngram = cv_ngram.transform(test_corpus)

In [16]:
print(f'After apply the BoW(Bag of Word) the shape: {X_train_bow.shape}')
print(f'Vocabulary size: {len(cv.vocabulary_)}')
# cv.get_feature_names_out()[1000]

After apply the BoW(Bag of Word) the shape: (4457, 6213)
Vocabulary size: 6213


### TF-IDF

In [17]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [18]:
len(tfidf.vocabulary_)

6213

In [19]:
new_text = ["Hello World how are you!"]

tfidf.transform(new_text).toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 6213))

In [121]:
new_text = ["Once upon a time in BD."]

tfidf.transform(new_text).toarray()



array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 6213))

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'System device: {device}')

System device: cuda


### Word2Vec

In [124]:
from gensim.models import KeyedVectors

wv_model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True, limit=500000)


FileNotFoundError: [Errno 2] No such file or directory: 'GoogleNews-vectors-negative300.bin.gz'

#### Convert to `Torch`

In [122]:

X_train = torch.tensor(X_train_tfidf, dtype=torch.long)
X_test = torch.tensor(X_test_tfidf, dtype=torch.long)


In [123]:
X_train.shape

torch.Size([4457, 6213])